# Hippocampus, novel maze with pre/post sleep (Dandiset 001695): exploration + Grassmannian pipeline

Part 1 keeps the exploration of [Dandiset 001695](https://dandiarchive.org/dandiset/001695/0.260319.2023)
with pynapple: one downloaded session, its units (CA1 / CA3 / RSC, with cell-type labels), position and
speed on the maze, sleep intervals, and the statistics of the binned counts.

Part 2 runs the standardised Grassmannian pipeline on that session in **full-recording** mode
(sleep and maze included; `T_RANGE="maze"` restricts it).

In [ ]:
# ============ storage and run naming: inputs and outputs both live under storage/ ============
from pathlib import Path

STORAGE_DIR   = Path("../../storage").resolve()                 # same storage root the downloads always used
DOWNLOAD_DIR  = STORAGE_DIR / "dandi_downloads"                  # INPUTS only (NWB files)
DATASET_LABEL = "Hp_001695"
RUN_NAME      = "basic_grassmanian"                              # name of this run; change it to keep runs apart
DATASET_DIR   = STORAGE_DIR / "grassmanian_test" / DATASET_LABEL     # dataset-level caches: region survey, loaded sessions
OUTPUT_DIR    = DATASET_DIR / RUN_NAME                               # this run's caches (window spectra, Gr mean), per session
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
# Only data that saves recomputation is written; figures are displayed, not saved.
print("inputs :", DOWNLOAD_DIR)
print("caches :", OUTPUT_DIR)

In [ ]:
from pathlib import Path
from dandi.dandiapi import DandiAPIClient
import pynapple as nap

# ---- config ----
DANDISET_ID = "001695"
VERSION_ID  = "0.260319.2023"
SUBJECT     = "sub-M01/"                                  # which subject to scan
DOWNLOAD_DIR = STORAGE_DIR / "dandi_downloads"             # set in the storage cell at the top
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)          # storage folder (create if missing)

# ---- discover sessions, pick smallest to keep it quick ----
with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, VERSION_ID)
    sessions = [(a.path, a.size)
                for a in dandiset.get_assets_with_path_prefix(SUBJECT, order="path")]
    # behavior+ecephys sessions are the ones with position data; prefer those
    behavior = [s for s in sessions if "behavior" in s[0]]
    pool = behavior or sessions                          # fall back to any if none flagged
    target_path = min(pool, key=lambda t: t[1])[0]       # smallest in the pool
    print("Picked:", target_path)

    # ---- download into storage (make nested sub-XX/ parent first) ----
    asset = dandiset.get_asset_by_path(target_path)
    asset_out = DOWNLOAD_DIR / target_path
    asset_out.parent.mkdir(parents=True, exist_ok=True)  # download() won't create dirs
    if not asset_out.exists():                           # skip re-download if already local
        asset.download(asset_out)
    print("Local file:", asset_out)

# ---- load in pynapple ----
data = nap.load_file(str(asset_out))
print("\nObjects in this NWB file:")
print(data)                                              # look here for position/behavior keys

# ---- find spatial-navigation data ----
# pynapple exposes NWB objects by key; spatial data is usually a TsdFrame of x/y
# common keys across datasets: "position", "SpatialSeries", "head_position", etc.
spatial_keys = [k for k in data.keys()
                if any(w in k.lower() for w in ["position", "spatial", "pos", "head"])]
print("\nCandidate spatial keys:", spatial_keys)

if spatial_keys:
    pos = data[spatial_keys[0]]                          # TsdFrame: columns are x/y (/z)
    print(f"\nUsing '{spatial_keys[0]}':")
    print(pos)
    print("Columns:", list(getattr(pos, "columns", [])))
    print("Time span (s):", pos.index[-1] - pos.index[0])
else:
    print("\nNo obvious spatial key — print(data) above to see actual object names.")

In [ ]:
import pynapple as nap
from pathlib import Path

asset_out = Path("../../storage/dandi_downloads/sub-M01/"
                 "sub-M01_ses-20240313T100000_behavior+ecephys.nwb").resolve()
data = nap.load_file(str(asset_out))

# ---- pynapple's view: top-level objects and their types ----
print("=" * 60)
print("PYNAPPLE OBJECTS")
print("=" * 60)
print(data)

# ---- units TsGroup: metadata columns + shape ----
units = data["units"]
print("\n" + "=" * 60)
print("UNITS (TsGroup)")
print("=" * 60)
print("n_units:", len(units))
print("metadata_columns:", list(units.metadata_columns))
print("\nfull metadata table:")
print(units.metadata)                     # every per-unit column, all rows

# ---- each non-units object: type, shape, columns, time span ----
print("\n" + "=" * 60)
print("OTHER OBJECTS")
print("=" * 60)
for key in data.keys():
    if key == "units":
        continue
    obj = data[key]
    line = f"{key:32s} {type(obj).__name__}"
    if hasattr(obj, "columns"):
        line += f"  cols={list(obj.columns)}"
    if hasattr(obj, "shape"):
        line += f"  shape={obj.shape}"
    if hasattr(obj, "index") and len(obj) > 0:
        line += f"  t=[{obj.index[0]:.1f}, {obj.index[-1]:.1f}]s"
    elif hasattr(obj, "start"):            # IntervalSet
        line += f"  n_intervals={len(obj)}"
    print(line)

# ---- raw NWB hierarchy (the underlying file, everything pynapple didn't surface) ----
print("\n" + "=" * 60)
print("RAW NWB TREE")
print("=" * 60)
try:
    import pynwb, h5py
    with pynwb.NWBHDF5IO(str(asset_out), mode="r") as io:
        nwb = io.read()
        print(nwb)                          # full NWB object: acquisition, processing, intervals, etc.
except Exception as e:
    print("Could not open raw NWB:", e)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# per-unit metadata available on the TsGroup (summary scores + labels, not spikes)
print("Metadata columns on units:", list(units.metadata_columns))

region_col = "cell_area"   # region label; 'cell_type' = narrow/wide interneuron etc.

regions = np.asarray(units.get_info(region_col))
cell_types = np.asarray(units.get_info("cell_type"))

# build a region x cell_type count table, then add a "Total" column
tab = pd.crosstab(pd.Series(regions, name="area"),
                  pd.Series(cell_types, name="type"))
tab.insert(0, "Total", tab.sum(axis=1))   # Total first, then each cell type
print("\nCounts (region x cell_type, with Total):")
print(tab)

# ---- grouped bar plot: regions on X, (Total + each cell_type) bars per region ----
areas = tab.index.tolist()          # X groups
cols = tab.columns.tolist()         # bars within each group: ['Total', type1, type2, ...]
n_bars = len(cols)
x = np.arange(len(areas))
width = 0.8 / n_bars                 # fit all bars within each region slot

fig, ax = plt.subplots(figsize=(1.6 * len(areas) + 3, 5))
cmap = plt.get_cmap("tab10")

for i, col in enumerate(cols):
    offsets = x + (i - (n_bars - 1) / 2) * width   # center the group on the tick
    bars = ax.bar(offsets, tab[col].values, width,
                  label=col, color=cmap(i))
    ax.bar_label(bars, padding=2, fontsize=8)      # count above each bar

ax.set_xticks(x); ax.set_xticklabels(areas)
ax.set(xlabel="brain region", ylabel="unit count",
       title="Unit counts by region (Total + per cell type)")
ax.legend(title="", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import pynapple as nap

# ---- reload objects from the file so the cell stands alone ----
session_nwb_path = Path("../../storage/dandi_downloads/sub-M01/"
                        "sub-M01_ses-20240313T100000_behavior+ecephys.nwb").resolve()
session_nwb  = nap.load_file(str(session_nwb_path))
spike_group  = session_nwb["units"]        # TsGroup: 240 units, spike times + per-unit metadata
position_xy  = session_nwb["Position"]      # TsdFrame x/y, maze epoch only

n_units_per_group = 5     # neurons to show per (region, cell_type) combo
raster_window_s   = 60    # seconds to display in the raster (short window so spikes are legible)

# ---- compute running speed from position (not reliably provided in the file) ----
position_time_s = position_xy.index.values
position_x_vals = position_xy["x"].values
position_y_vals = position_xy["y"].values
# instantaneous speed = distance between consecutive samples / dt  (units: position-units / s)
dt_s = np.diff(position_time_s)
dxy = np.sqrt(np.diff(position_x_vals)**2 + np.diff(position_y_vals)**2)
speed_vals = dxy / dt_s
speed_time_s = position_time_s[1:]               # speed is defined between samples
# light smoothing over ~0.25 s to reduce sample-to-sample jitter
smooth_n = max(1, int(0.25 / np.median(dt_s)))
speed_vals_smooth = np.convolve(speed_vals, np.ones(smooth_n)/smooth_n, mode="same")

# ---- per-unit metadata arrays ----
unit_brain_regions = np.asarray(spike_group.get_info("cell_area"))   # CA1 / CA3 / RSC
unit_cell_types    = np.asarray(spike_group.get_info("cell_type"))   # Narrow/Wide Interneuron, Pyramidal Cell
unit_id_keys       = np.asarray(spike_group.index)                   # 1-based unit keys

cell_type_names  = sorted(set(unit_cell_types))
cell_type_colors = {ct: c for ct, c in zip(cell_type_names, plt.get_cmap("tab10").colors)}

# ---- pick up to n neurons per region x cell_type ----
selected_units = []   # (unit_id, brain_region, cell_type)
for brain_region in sorted(set(unit_brain_regions)):
    for cell_type in cell_type_names:
        matching_unit_ids = unit_id_keys[(unit_brain_regions == brain_region)
                                         & (unit_cell_types == cell_type)]
        for unit_id in matching_unit_ids[:n_units_per_group]:
            selected_units.append((unit_id, brain_region, cell_type))

# ---- display window: first raster_window_s of the maze epoch ----
window_start_s = position_time_s[0]
window_end_s   = window_start_s + raster_window_s

fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True,
                         gridspec_kw={"height_ratios": [len(selected_units)/8, 1, 1, 1]})

# ---- panel 0: spike raster (windowed) ----
raster_ax = axes[0]
for raster_row, (unit_id, brain_region, cell_type) in enumerate(selected_units):
    unit_spike_times = spike_group[unit_id].get(window_start_s, window_end_s)
    raster_ax.plot(unit_spike_times.index.values,
                   np.full(len(unit_spike_times), raster_row), "|",
                   color=cell_type_colors[cell_type], markersize=4, mew=0.6)
raster_ax.set_yticks(range(len(selected_units)))
raster_ax.set_yticklabels([f"{r[:4]}·{c[:4]}" for _, r, c in selected_units], fontsize=6)
raster_ax.set(ylabel="unit (region·cell type)",
              title=f"Spike rasters — up to 5 units per cell type per region "
                    f"({raster_window_s}s window)")
raster_ax.invert_yaxis()
raster_ax.legend(handles=[Line2D([0],[0], color=cell_type_colors[ct], lw=2, label=ct)
                          for ct in cell_type_names],
                 fontsize=8, loc="upper right", ncol=len(cell_type_names))

# ---- panels 1 & 2: position x and y ----
axes[1].plot(position_time_s, position_x_vals, lw=0.6, color="tab:blue")
axes[1].set(ylabel="x position (cm)")
axes[2].plot(position_time_s, position_y_vals, lw=0.6, color="tab:purple")
axes[2].set(ylabel="y position (cm)")

# ---- panel 3: computed running speed ----
axes[3].plot(speed_time_s, speed_vals_smooth, lw=0.6, color="k")
axes[3].set(ylabel="speed (cm/s)", xlabel="time (s)")

axes[0].set_xlim(window_start_s, window_end_s)   # all panels locked to the window
plt.tight_layout(); plt.show()

print(f"Showing {len(selected_units)} units | window {window_start_s:.0f}–{window_end_s:.0f} s")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# reuse objects already loaded above
sleep_intervals = session_nwb["SleepStates"]
speed_tsd = session_nwb["Speed"]
position_tsd = session_nwb["Position"]
units_tsgroup = session_nwb["units"]

# ---- helper to get overall spike-data bounds ----
all_starts = []
all_ends = []
for unit_id in units_tsgroup.index:
    spike_times = units_tsgroup[unit_id].index.values
    if len(spike_times):
        all_starts.append(spike_times[0])
        all_ends.append(spike_times[-1])

units_start = min(all_starts)
units_end = max(all_ends)

position_start = position_tsd.index[0]
position_end = position_tsd.index[-1]

speed_start = speed_tsd.index[0]
speed_end = speed_tsd.index[-1]

print("Overall recording bounds")
print(f"units    : {units_start:.2f} to {units_end:.2f} s  ({units_end - units_start:.2f} s)")
print(f"position : {position_start:.2f} to {position_end:.2f} s  ({position_end - position_start:.2f} s)")
print(f"speed    : {speed_start:.2f} to {speed_end:.2f} s  ({speed_end - speed_start:.2f} s)")

print("\nSleep intervals")
sleep_starts = np.asarray(sleep_intervals.start)
sleep_ends = np.asarray(sleep_intervals.end)
for i, (s, e) in enumerate(zip(sleep_starts, sleep_ends), start=1):
    print(f"{i:02d}: {s:.2f} to {e:.2f} s  ({e - s:.2f} s)")

# ---- quick overlap summary ----
print("\nBehavior epoch likely covered by Position/Speed")
print(f"behavior : {max(position_start, speed_start):.2f} to {min(position_end, speed_end):.2f} s")

# ---- timeline plot ----
fig, ax = plt.subplots(figsize=(14, 3))

# full spike recording
ax.plot([units_start, units_end], [3, 3], lw=8, color="0.7", solid_capstyle="butt", label="units")

# position/speed coverage
ax.plot([position_start, position_end], [2, 2], lw=8, color="tab:blue", solid_capstyle="butt", label="Position")
ax.plot([speed_start, speed_end], [1, 1], lw=8, color="tab:green", solid_capstyle="butt", label="Speed")

# sleep intervals
for j, (s, e) in enumerate(zip(sleep_starts, sleep_ends)):
    ax.plot([s, e], [0, 0], lw=8, color="tab:red", solid_capstyle="butt",
            label="SleepStates" if j == 0 else None)

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["SleepStates", "Speed", "Position", "Units"])
ax.set_xlabel("time (s)")
ax.set_title("Recording timeline: behavior vs sleep coverage")
ax.legend(loc="upper right")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm
import copy

# ---------- Inputs ----------
pos = session_nwb["Position"]
has_speed = "Speed" in session_nwb.keys()
spd_obj = session_nwb["Speed"] if has_speed else None

# ---------- Build behavior dataframe ----------
t = np.asarray(pos.index.values, dtype=float)
x = np.asarray(pos["x"].values, dtype=float)
y = np.asarray(pos["y"].values, dtype=float)

good = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
t, x, y = t[good], x[good], y[good]

dt = np.diff(t, prepend=t[0])
dx = np.diff(x, prepend=x[0])
dy = np.diff(y, prepend=y[0])

speed_pos = np.zeros_like(t)
mask = dt > 0
speed_pos[mask] = np.sqrt(dx[mask]**2 + dy[mask]**2) / dt[mask]

if has_speed:
    ts = np.asarray(spd_obj.index.values, dtype=float)
    vs = np.asarray(spd_obj.values, dtype=float)
    ok = np.isfinite(ts) & np.isfinite(vs)
    ts, vs = ts[ok], vs[ok]
    speed = np.interp(t, ts, vs) if len(ts) > 1 else speed_pos
else:
    speed = speed_pos

speed = np.clip(speed, 0, None)

df = pd.DataFrame({"t": t, "x": x, "y": y, "speed": speed}).dropna().reset_index(drop=True)

# ---------- Speed cap: mean + 3 SD ----------
mu = df["speed"].mean()
sd = df["speed"].std(ddof=1)
cap = mu + 3 * sd
df["speed_clip"] = np.minimum(df["speed"], cap)
df["is_over_cap"] = df["speed"] > cap

# ---------- Diagnostic stats ----------
def skewness(v):
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < 3:
        return np.nan
    m = v.mean()
    s = v.std(ddof=0)
    if s == 0:
        return 0.0
    return np.mean(((v - m) / s) ** 3)

eps = 1e-6
raw_skew = skewness(df["speed"].values)
log_skew = skewness(np.log(df["speed"].values + eps))

stats = pd.DataFrame({
    "metric": [
        "n_samples", "duration_s", "mean_speed", "std_speed", "median_speed",
        "p95_speed", "p99_speed", "max_speed", "speed_cap_mean_plus_3sd",
        "frac_above_cap", "raw_skew", "log_skew"
    ],
    "value": [
        len(df),
        df["t"].iloc[-1] - df["t"].iloc[0],
        df["speed"].mean(),
        df["speed"].std(ddof=1),
        df["speed"].median(),
        df["speed"].quantile(0.95),
        df["speed"].quantile(0.99),
        df["speed"].max(),
        cap,
        df["is_over_cap"].mean(),
        raw_skew,
        log_skew
    ]
})
print(stats)

# ---------- Plot 1: speed distribution ----------
v = df["speed"].values
vpos = v[v > 0]
if len(vpos) == 0:
    raise ValueError("No positive speed samples available.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

# Left: full histogram + fits
ax[0].hist(v, bins=80, density=True, alpha=0.35, color="gray", label="speed histogram")
ax[0].axvline(cap, color="red", ls="--", lw=2, label=f"cap = mean+3SD ({cap:.2f})")

m, s = np.mean(v), np.std(v, ddof=0) + 1e-12
grid = np.linspace(0, max(v.max(), cap) * 1.1, 600)
normal_pdf = (1 / (s * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((grid - m) / s) ** 2)

lv = np.log(vpos + eps)
ml, sl = lv.mean(), lv.std(ddof=0) + 1e-12
logn_pdf = (1 / ((grid + eps) * sl * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((np.log(grid + eps) - ml) / sl) ** 2)

ax[0].plot(grid, normal_pdf, color="tab:blue", lw=2, label="normal fit")
ax[0].plot(grid, logn_pdf, color="tab:green", lw=2, label="log-normal fit")
ax[0].set_title("Speed distribution")
ax[0].set_xlabel("Speed")
ax[0].set_ylabel("Density")
ax[0].legend()

# Right: log-x, no shaded histogram
counts, bins = np.histogram(vpos, bins=80, density=True)
centers = 0.5 * (bins[:-1] + bins[1:])
ax[1].plot(centers, counts, color="0.4", lw=1.8, label="speed>0 density")
ax[1].plot(grid[grid > 0], normal_pdf[grid > 0], color="tab:blue", lw=2, label="normal fit")
ax[1].plot(grid[grid > 0], logn_pdf[grid > 0], color="tab:green", lw=2, label="log-normal fit")
ax[1].set_xscale("log")
ax[1].set_title("Speed distribution (log-x)")
ax[1].set_xlabel("Speed (log scale)")
ax[1].set_ylabel("Density")
ax[1].legend()

plt.tight_layout()
plt.show()

# ---------- Colormap + helper (fixed error, no cm.get_cmap) ----------
base_cmap = plt.get_cmap("viridis")
cmap = base_cmap.copy() if hasattr(base_cmap, "copy") else copy.copy(base_cmap)
cmap.set_under("blue")   # speed == 0
cmap.set_over("red")     # speed > cap
norm = LogNorm(vmin=1e-3, vmax=cap)

def colored_line_log(ax, xseg, yseg, speedseg, cmap_obj, norm_obj):
    pts = np.array([xseg, yseg]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    c = np.asarray(speedseg[:-1], dtype=float)
    c = np.where(np.isfinite(c), c, 0.0)
    c = np.where(c > cap, cap * 1.0001, c)  # force over-range to "red"
    lc = LineCollection(segs, cmap=cmap_obj, norm=norm_obj)
    lc.set_array(c)
    lc.set_linewidth(2.0)
    ax.add_collection(lc)
    return lc

# ---------- Plot 2: full trajectory (no colorbar) ----------
fig, ax = plt.subplots(figsize=(7, 7))
_ = colored_line_log(ax, df["x"].values, df["y"].values, df["speed"].values, cmap, norm)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Maze trajectory heatmapped by speed\n(blue=0, red>mean+3SD)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ---------- Plot 3: 10 temporal epochs with gray history (no colorbar) ----------
n_epochs = 10
idx_edges = np.linspace(0, len(df)-1, n_epochs+1).astype(int)

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(n_epochs):
    ax = axes[k]
    i0, i1 = idx_edges[k], idx_edges[k+1]
    if i1 <= i0 + 2:
        ax.set_title(f"Epoch {k+1}: too short")
        continue

    if i0 > 1:
        ax.plot(df["x"].values[:i0], df["y"].values[:i0], color="0.7", lw=1.2, alpha=0.8)

    xk = df["x"].values[i0:i1]
    yk = df["y"].values[i0:i1]
    sk = df["speed"].values[i0:i1]
    _ = colored_line_log(ax, xk, yk, sk, cmap, norm)

    t0, t1 = df["t"].values[i0], df["t"].values[i1-1]
    ax.set_title(f"Epoch {k+1}\n{t0:.1f}-{t1:.1f}s")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)

fig.suptitle("Trajectory across 10 behavior epochs\n(gray = previous trajectory, color = current speed)", y=1.02)
for a in axes:
    a.set_xlabel("x")
    a.set_ylabel("y")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BIN = 0.050

unit_ids = np.asarray(spike_group.index)
spike_arrays = [spike_group[u].index.values for u in unit_ids]

t_start = min(s[0] for s in spike_arrays if len(s))
t_end   = max(s[-1] for s in spike_arrays if len(s))
edges = np.arange(t_start, t_end + BIN, BIN)
bin_times_s = edges[:-1]
T, N = len(edges) - 1, len(unit_ids)

# histogram does the binning (fast bin-search); one call per neuron
counts = np.stack([np.histogram(spk, bins=edges)[0] for spk in spike_arrays], axis=1).astype(float)
neuron_totals = np.array([spk.size for spk in spike_arrays], dtype=float)

# vectorized divide-by-total then z-score
neuron_totals[neuron_totals == 0] = 1
prop = counts / neuron_totals
spike_matrix_norm = (prop - prop.mean(0)) / np.where(prop.std(0) == 0, 1, prop.std(0))

print(f"matrix: {spike_matrix_norm.shape} | NaN/inf:",
      np.isnan(spike_matrix_norm).any(), np.isinf(spike_matrix_norm).any())

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
ax[0,0].hist(spike_matrix_norm.ravel(), bins=100, log=True)
ax[0,0].set(title="Normalized values (log y)", xlabel="z", ylabel="freq")
frac_zero = (counts == 0).mean(0)
ax[0,1].hist(frac_zero, bins=30); ax[0,1].set(title="Fraction empty bins/neuron", xlabel="P(count=0)")
m, v = counts.mean(0), counts.var(0)
ax[0,2].loglog(m+1e-3, v+1e-6, ".", alpha=0.4)
lims=[m.min()+1e-3, m.max()]; ax[0,2].plot(lims, lims, "k--", label="var=mean")
ax[0,2].set(title="Mean vs var — raw", xlabel="mean", ylabel="var"); ax[0,2].legend()
ax[1,0].hist(spike_matrix_norm.std(0), bins=30); ax[1,0].set(title="Per-neuron std after norm (~1)")
lo = np.argsort(neuron_totals)[N//4]
ax[1,1].hist(spike_matrix_norm[:, lo], bins=50); ax[1,1].set(title=f"Neuron {lo}: normalized dist")
ax[1,2].hist(neuron_totals, bins=40); ax[1,2].set(title="Total spikes/neuron (divisor)")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BIN = 0.050
binned = spike_group.count(BIN)          # TsdFrame [T x N], raw counts
counts = binned.values.astype(float)     # T bins x N neurons
firing_rates = counts.mean(0) / BIN      # per-neuron mean rate (Hz)

fig, ax = plt.subplots(2, 3, figsize=(16, 9))

# 1) distribution of raw counts across ALL neurons/bins — is it spiky at 0?
ax[0,0].hist(counts.ravel(), bins=np.arange(0, counts.max()+2)-0.5, log=True)
ax[0,0].set(title="Raw spike counts per bin (log y)", xlabel="count", ylabel="freq")

# 2) sparsity: fraction of bins that are ZERO, per neuron
frac_zero = (counts == 0).mean(0)
ax[0,1].hist(frac_zero, bins=30)
ax[0,1].set(title="Fraction of empty bins per neuron", xlabel="P(count=0)", ylabel="# neurons")

# 3) mean vs variance per neuron — Poisson sits on y=x; Gaussian wants flat
unit_mean = counts.mean(0); unit_var = counts.var(0)
ax[0,2].loglog(unit_mean, unit_var, ".", alpha=0.4)
lims = [unit_mean.min()+1e-3, unit_mean.max()]
ax[0,2].plot(lims, lims, "k--", label="var=mean (Poisson)")
ax[0,2].set(title="Mean vs variance per neuron", xlabel="mean", ylabel="variance"); ax[0,2].legend()

# 4) per-neuron firing-rate spread — how heterogeneous are the scales?
ax[1,0].hist(firing_rates, bins=40)
ax[1,0].set(title="Per-neuron firing rate", xlabel="Hz", ylabel="# neurons")

# 5) example single-neuron count distribution (a low-rate one) raw vs sqrt
lo = np.argsort(firing_rates)[len(firing_rates)//4]   # a lowish-rate neuron
ax[1,1].hist(counts[:, lo], bins=30, alpha=0.6, label="raw")
ax[1,1].hist(np.sqrt(counts[:, lo]), bins=30, alpha=0.6, label="sqrt")
ax[1,1].set(title=f"Neuron {lo}: raw vs sqrt", xlabel="value"); ax[1,1].legend()

# 6) does z-scoring equalize scales? std per neuron after z-score should be ~1
zc = (counts - counts.mean(0)) / np.where(counts.std(0)==0, 1, counts.std(0))
ax[1,2].hist(zc.std(0), bins=30)
ax[1,2].set(title="Per-neuron std after z-score", xlabel="std (should be ~1)", ylabel="# neurons")

plt.tight_layout(); plt.show()

print(f"median firing rate: {np.median(firing_rates):.2f} Hz")
print(f"median P(empty bin): {np.median(frac_zero):.2f}")
print(f"mean/var correlation: overdispersed if var >> mean")

## Standardised Grassmannian pipeline

Everything from here on is shared with the other real-data notebook through
`riemann_spiking_analysis.py` (one folder up). The object
`riemann_spiking_analysis_obj`:

1. **holds sessions** (`session_data`): registered = *available to load*, `loaded` = tables in memory;
   `status()` lists both. Loaders turn a dandi asset / open NWB file into the same tables for every
   dataset: `units` (`unit_id, spike_times, region, cell_type, quality, n_spikes, rate_hz`),
   `trials` (if any), `epochs`, `events`.
2. **preprocesses with functions you pass in**: `unit_filter`, `trial_filter`, `count_transform`,
   `normalize` (default: z-score each neuron across all retained bins), `k_rule`.
3. **bins** spikes at `count_bin_ms`, inside trials (`mode="trials"`) or over the recording (`mode="full"`).
4. **one covariance matrix per window** of `mollification_bin_ms`, stepped every
   `mollification_step_ms`, never crossing a trial / epoch boundary; full eigendecomposition per window.
   Explained variance = each window's eigenvalues divided by their sum. Participation ratio =
   (Σλ)² / Σλ².
5. **k**: `k_median_elbow()` takes the elbow (farthest point below the chord) of every window's
   explained-variance curve and uses the median across windows; `k_fixed(K)` is the alternative.
6. **Gr(k, N)**: the top-k eigenvectors of each window are a point on the Grassmannian. Karcher mean
   of all windows, log map at the mean, PCA of the tangent vectors; the figure shows the top-2 tangent
   directions flat (tangent plane) and wrapped onto a sphere by the exponential map at the pole
   (polar angle = projected tangent norm / √k, an RMS principal angle, so the equator is the largest
   distance the manifold allows). Bottom: the geodesic angle between consecutive windows. No clustering yet.

Ambiguities settled here (change if you meant otherwise): *mollification bin* = the window a covariance
matrix runs over, with a separate `count_bin_ms` for the spike-count resolution inside it; windows are
z-scored globally (per neuron over all retained bins) and then centred per window before the covariance;
the sphere / plane views are projections of a k(N−k)-dimensional tangent space, the % of tangent
variance they carry is printed on the axes.

In [ ]:
# ---- shared pipeline module: RealData_Grassmanian_Tests/riemann_spiking_analysis.py ----
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _find_module_dir(name="riemann_spiking_analysis.py"):
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / name).exists():
            return d
    raise FileNotFoundError("%s not found in or above %s (git pull the repo on this machine)" % (name, here))


MODULE_DIR = _find_module_dir()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import riemann_spiking_analysis as rsa
importlib.reload(rsa)                     # picks up edits to the module without restarting the kernel
from riemann_spiking_analysis import (RiemannSpikingAnalysis, load_alm_000363, load_hp_001695,
                                      zscore_units, sqrt_counts, identity, keep_all,
                                      k_median_elbow, k_fixed, session_key_from_path)
print("module:", MODULE_DIR / "riemann_spiking_analysis.py")

In [ ]:
# ============ preprocessing decisions: every step is a function handed to the object ============
HP_REGIONS            = ["CA1", "CA3"]   # units kept by cell_area; None = all (CA1, CA3, RSC)
MIN_RATE_HZ           = 0.1              # drop near-silent units
COUNT_BIN_MS          = 25.0             # spike-count (mollifier) bin
MOLLIFICATION_BIN_MS  = 1000.0           # how long counts are accumulated for ONE covariance matrix = one point on Gr(k, N)
MOLLIFICATION_STEP_MS = 500.0            # stride between windows; None = window length (non-overlapping)
T_RANGE               = None             # None = whole recording; "maze" / "sleep" = those epochs only; or (t0, t1) in s
K_STORE               = 15               # eigenvectors kept per window; k can be anything <= this without recomputing


def hp_unit_filter(units):
    """Keep units of HP_REGIONS firing at least MIN_RATE_HZ.
    units columns: unit_id, spike_times, region (cell_area), cell_type, quality, n_spikes, rate_hz + the
    other per-unit metadata columns of the file (burstIndex, cv2, troughToPeak, ...)."""
    keep = units["rate_hz"] >= MIN_RATE_HZ
    if HP_REGIONS is not None:
        keep &= units["region"].isin(HP_REGIONS)
    return keep.values


riemann_spiking_analysis_obj = RiemannSpikingAnalysis(
    mode="full",                              # hippocampus: the whole recording is one stretch (cut by T_RANGE if set)
    t_range=T_RANGE,
    count_bin_ms=COUNT_BIN_MS,
    mollification_bin_ms=MOLLIFICATION_BIN_MS,
    mollification_step_ms=MOLLIFICATION_STEP_MS,
    unit_filter=hp_unit_filter,
    trial_filter=keep_all,                    # no trials in this dataset
    count_transform=identity,                 # alternatives: sqrt_counts, or any f(counts) -> array
    normalize=zscore_units,                   # z-score each neuron across all retained bins (for now)
    k_rule=k_median_elbow(m=50),              # alternative: k_fixed(6)
    k_store=K_STORE,
    run_name=RUN_NAME,
    output_dir=OUTPUT_DIR,                         # caches window spectra + Gr mean; reused when data/settings match
    session_cache_dir=DATASET_DIR / "session_cache",   # caches loaded spike trains / trials, shared by all runs
    use_cache=True,                                # False = always recompute
    save_figures=False,                            # figures are only displayed
)
riemann_spiking_analysis_obj.describe()

In [ ]:
# ============ sessions: what is available vs what is in memory ============
obj = riemann_spiking_analysis_obj
SESSION_KEY = session_key_from_path(asset_out)            # local file from the exploration cells, e.g. 'M01_20240313T100000'

# the session already downloaded above is registered with its local file
obj.add_session(SESSION_KEY, loader=load_hp_001695, local_path=asset_out, download_dir=DOWNLOAD_DIR)

# the other behavior+ecephys sessions are registered as *available*; a file is downloaded into DOWNLOAD_DIR
# only when obj.load(key) is called for it
obj.register_dandiset(DANDISET_ID, VERSION_ID, load_hp_001695,
                      path_filter=lambda p: "behavior+ecephys" in p, download_dir=DOWNLOAD_DIR)

obj.load(SESSION_KEY)                                     # -> standard units / epochs tables (no trials here)
status = obj.status()
print("%d sessions registered, %d loaded" % (len(status), int(status["loaded"].sum())))
display(status)
rec = obj.session_data[SESSION_KEY]
print("epochs:", {k: v.shape for k, v in rec.epochs.items()}, "| t_range (s):", rec.t_range)
display(pd.crosstab(rec.units["region"], rec.units["cell_type"]))

In [ ]:
# ============ 1. filter units / trials, cut into segments, bin at count_bin_ms, normalise ============
st = obj.prepare(SESSION_KEY)
print(st.units["region"].value_counts().to_string())
print("X (bins x units):", st.X.shape, "| segments with >= 1 bin:", len(np.unique(st.bin_seg)))

In [ ]:
# ============ 2. one covariance per window + full eigendecomposition;  3. choose k ============
windows = obj.window_spectra()
k = obj.choose_k()                      # uses k_rule from the config cell; e.g. obj.choose_k(k_fixed(6)) to override
display(windows.head())
display(obj.summary())

In [ ]:
# ============ spectra: participation ratio, explained variance (top 10, violins), raw eigenvalues ============
obj.plot_participation_ratio(); plt.show()
obj.plot_explained_variance(n_show=10); plt.show()      # lambda_i / sum(lambda) per component, median traced; not cumulative
obj.plot_eigenvalue_spectra(); plt.show()               # every window's spectrum over all dimensions, coloured by time

In [ ]:
# ============ 4. Gr(k, N): Karcher mean, log map at the mean, tangent PCA, subspace velocity ============
obj.fit_grassmann(mean="karcher")       # mean="extrinsic" is faster and gives the same picture to first order
# per-window results now on st: st.distance (geodesic distance to the mean), st.angles (principal angles),
# st.tangent_coords (top tangent PCs), st.velocity (angle between consecutive windows), st.U, st.mu

In [ ]:
# ============ 5. sphere + tangent plane + timeline ============
# with ipympl installed, `%matplotlib widget` at the top of the notebook makes the sphere drag-to-rotate;
# otherwise turn it with view=(elev, azim)
fig = obj.plot_grassmann(view=(28, -55))                           # colour = time
plt.show()
fig = obj.plot_grassmann(labels=obj.window_labels("epochs"))       # colour = sleep / maze / other
plt.show()

### Knobs worth turning next

* `MOLLIFICATION_BIN_MS` vs `COUNT_BIN_MS`: a window of L count bins has covariance rank ≤ L − 1, so
  with more units than bins per window the spectra are truncated (the window-spectra printout says so).
* `k_rule`: `k_median_elbow(m=..)` (elbow of the first m components) or `k_fixed(K)`; `K_STORE`
  bounds k without recomputing the spectra.
* `normalize` / `count_transform`: hand in any `f(array) -> array`, e.g. `sqrt_counts` before the z-score.
* `obj.plot_grassmann(view=(elev, azim))` to turn the sphere; with `ipympl` installed,
  `%matplotlib widget` at the top makes it drag-to-rotate.
* Caches: change `RUN_NAME` for a fresh run folder, `use_cache=False` to force recomputation,
  `obj.load(key, refresh=True)` to re-read a session from its source.
* Other sessions: `obj.status()` lists what is registered; `obj.load(key)` then `obj.prepare(key)` …
  run the same cells on it; `obj.unload(key)` frees memory.